# Goal 4 -- Autonomous Compliance Sentinel Agent
### SRH Heidelberg | Data Ethics and Responsible AI
**Authors:** Vikrant Singh and Kay Marc Muller

---

## What is this notebook?

This notebook builds the **Goal 4 Agent**, the final and most important piece of the Autonomous Compliance Sentinel project.

The system fetches internal project proposals directly from **Jira**, automatically checks whether they violate any of our 9 Responsible AI policies (RAI-01 through RAI-09), and writes the compliance findings back to Jira as a comment. The human reviewer then reads the findings and decides what remediation action to take.

**The agent identifies the problem. The human decides the fix.**

---

## The full pipeline at a glance

```
Jira Project (DE2)
        |
        v
fetch_proposals()  -- fetch all issues and descriptions ONCE at startup
        |
        v
User picks one proposal by number from the printed list
        |
        v
[Signal 1] TF-IDF + Logistic Regression Classifier
        |
        | -- Compliant? --> post Compliant comment to Jira. Done.
        |
        | -- Red Flag? --> continue to Signal 2
        v
[Signal 2] LLM Agent + 6 Tools (RAG, XAI, Severity, Audit, Verdict, Narrative)
        |
        v
Human-in-the-Loop Gate
        | -- High severity  --> human types APPROVE or REJECT
        | -- Medium severity --> auto-approved, no human needed
        v
Post findings to Jira as a comment + save result to JSON
(Human reads findings and decides the fix separately)
```

---

## Two signals, not one

| Signal | What it is | When it runs |
|--------|-----------|--------------|
| Signal 1 | TF-IDF + Logistic Regression, threshold=0.4 | Always, on every proposal, fast |
| Signal 2 | LLM + 6 tools (RAG, XAI, severity, audit, verdict, narrative) | Only when Signal 1 flags Red Flag |

Signal 1 is fast and cheap. It runs in under 1 second and never calls the LLM. Signal 2 is slower but capable of nuanced, policy-grounded reasoning. Together they are more reliable and more auditable than either alone.

---

## The 9 RAI Policies

| Policy | Name | Severity |
|--------|------|----------|
| RAI-01 | Data Protection | High |
| RAI-02 | Transparency | High |
| RAI-03 | Fairness | High |
| RAI-04 | Human Dignity and Vulnerable Groups | High |
| RAI-05 | Prohibited Purpose | High |
| RAI-06 | Security | Medium |
| RAI-07 | Human Oversight | Medium |
| RAI-08 | Data Minimisation | Medium |
| RAI-09 | Explainability | Medium |

High severity violations (RAI-01 to RAI-05) are routed to a human reviewer who must type APPROVE or REJECT before any comment is posted.
Medium severity violations (RAI-06 to RAI-09) are auto-approved and proceed without human input.

---

## The 6 Tools

| Tool | Role | EU AI Act |
|------|------|-----------|
| `search_policies` | RAG: find relevant policy clauses from the catalogue | -- |
| `get_trigger_words` | XAI: which words in this proposal drove the classifier | Article 14 |
| `get_policy_severity` | Fixed lookup: High or Medium | Article 14 |
| `log_decision` | Audit trail: logs every decision with full evidence | Article 12 |
| `write_compliance_verdict` | Structured findings: policy, severity, reason | Article 12 |
| `llm_assessment` | Free-form narrative: plain language explanation for the reviewer | -- |

---
## Step 0 -- Install dependencies

Run this cell **once** when setting up a new environment. It installs every library this notebook needs.

You do not need to rerun this cell when restarting the kernel. The libraries are installed into your virtual environment and persist across sessions.

In [ ]:
%pip install langchain langgraph langchain-community langchain-chroma langchain-huggingface langchain-groq sentence-transformers pdfplumber requests python-dotenv

---
## Step 1 -- Imports and environment variables

We load the `.env` file **first**, before any other import, using `load_dotenv(override=True)`. The `override=True` flag forces a fresh reload even if environment variables were already set in a previous cell. This ensures every cell below always reads the latest values from `.env`.

All secrets live in `.env` and are never hardcoded in this notebook:
- `GROQ_API_KEY` -- the Groq API key for the LLM
- `JIRA_SERVER` -- your Atlassian site URL, e.g. `https://yourname.atlassian.net`
- `JIRA_EMAIL` -- your Atlassian account email address
- `JIRA_API_TOKEN` -- your personal Atlassian API token (starts with `ATATT`)
- `JIRA_PROJECT_KEY` -- the Jira project key, e.g. `DE2`

**Group 1 -- Signal 1 (the ML classifier)**
We need `joblib` to reload the already-trained pipeline from disk. We import `sklearn` pieces and `spacy` because `joblib` needs to find the class definitions (`ThresholdAdjustor`, `token_pos`) in memory before it can unpack the saved `.pkl` file. We do NOT retrain the model here. The model was trained in Goal 2 and saved to disk.

**Group 2 -- PDF loading and chunking**
`PDFPlumberLoader` reads the policy catalogue PDF page by page. `RecursiveCharacterTextSplitter` cuts those pages into smaller chunks that fit inside a vector search.

**Group 3 -- Embedding and vector storage**
`HuggingFaceEmbeddings` turns text chunks into numbers (vectors) that can be searched by meaning. `Chroma` stores those vectors on disk so we do not have to re-embed every time the kernel restarts.

**Group 4 -- Tools and LLM**
`tool` is a LangChain decorator that turns a plain Python function into something an LLM can call by name. `ChatGroq` connects to the Groq API to run `llama-3.1-8b-instant` on Groq's servers. `HumanMessage` and `SystemMessage` are LangChain message types the LLM understands.

**Group 5 -- Agent (LangGraph)**
`StateGraph`, `START`, `END` build the graph structure. `ToolNode` is a prebuilt LangGraph node that reads tool call requests from the LLM and executes the matching function. `MemorySaver` saves the graph state in memory so the human approval gate can pause and resume. `TypedDict`, `Annotated`, `operator` define the shared memory structure of the graph.

**Group 6 -- Jira integration and utilities**
`requests` makes HTTP calls to the Jira REST API v3. `HTTPBasicAuth` handles email and token authentication. `json`, `datetime`, `time` are standard Python utilities.

In [ ]:
# Load environment variables FIRST -- before any other import
from dotenv import load_dotenv
import os
load_dotenv(r"C:\MyGithubSpace\Data-Ethics\.env", override=True)

# Group 1 -- Signal 1: reloading the already-trained classifier
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
import spacy

# Group 2 -- PDF loading and chunking
from langchain_community.document_loaders import PDFPlumberLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Group 3 -- Embedding and vector storage
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Group 4 -- Tools and LLM
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage

# Group 5 -- Agent (LangGraph)
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated
import operator

# Group 6 -- Jira integration and utilities
import requests
from requests.auth import HTTPBasicAuth
import json
import datetime
import time

print("All imports OK")
print("GROQ key loaded   :", bool(os.environ.get("GROQ_API_KEY")))
print("Jira server loaded:", bool(os.environ.get("JIRA_SERVER")))

---
## Step 2a -- Reload Signal 1 (the trained classifier)

### What is Signal 1?

Signal 1 is the TF-IDF + Logistic Regression pipeline we trained in Goal 2. It was saved to disk as `logRegpipelineV2.pkl` using `joblib`. We do NOT retrain it here. We load it back from disk exactly as it was when training finished. This guarantees that the same weights produce the same predictions every time -- which is a requirement for Article 12 reproducibility.

### Why do we import from xai2.py first?

When `joblib` saved the pipeline, it did not save the *code* of `ThresholdAdjustor` or `token_pos`. It saved a *pointer* saying "find a class with this name in memory when loading." If those names do not exist in memory, `joblib.load()` will crash with `AttributeError`.

Importing from `xai2.py` puts those names into memory so `joblib` can find them when unpickling.

We also import `explain_prediction` and `audit_trail` from `xai2.py` because two of our tools (`get_trigger_words` and `log_decision`) call these functions internally.

### What is ThresholdAdjustor?

It is a custom wrapper around Logistic Regression that applies a custom decision threshold. The default sklearn threshold is 0.5: "classify as Red Flag if probability > 0.5." Our threshold is 0.4: "classify as Red Flag if probability > 0.4." This lowers the bar for flagging, which means fewer real violations get missed (higher recall). In a compliance system, missing a real violation is worse than a false alarm.

### What does the pipeline contain?

The loaded `classifier_pipeline` is a sklearn `Pipeline` object with two named steps:
- `"tfidf"` -- the `TfidfVectorizer` fitted on the training vocabulary, with the spaCy POS-filtered tokenizer
- `"clf"` -- the `ThresholdAdjustor` wrapping the trained `LogisticRegression`

Accessing individual steps: `classifier_pipeline.named_steps["tfidf"]` and `classifier_pipeline.named_steps["clf"]`

In [ ]:
from xai2 import ThresholdAdjustor, token_pos, nlp, explain_prediction, audit_trail

classifier_pipeline = joblib.load("logRegpipelineV2.pkl")
print("Signal 1 loaded.")
print("Pipeline steps:", list(classifier_pipeline.named_steps.keys()))

---
## Step 2b -- Load Signal 2 (the RAI Policy Catalogue PDF)

### What is Signal 2?

Signal 2 is the LLM agent. It only runs when Signal 1 flags a proposal as Red Flag. Its job is to reason about *which* policy was violated and *why*, using 6 tools that retrieve evidence from the policy catalogue, explain the classifier's decision, and write a structured audit trail.

### What is the RAI Policy Catalogue?

The RAI Policy Catalogue is the knowledge base Signal 2 reasons over. It contains 9 policies (RAI-01 through RAI-09), each written in a fixed structure every time:
- **Policy ID and name**
- **Severity** (High or Medium)
- **Regulatory Basis** (which EU AI Act article or GDPR provision applies)
- **Requirement** (what the proposal must include)
- **Violation Patterns** (real examples of what a violating proposal looks like)
- **Compliant Example** (a concrete example of a proposal that meets the requirement)

This fixed structure is what makes retrieval reliable: any chunk pulled from the catalogue contains enough context to stand alone.

### What does PDFPlumberLoader do?

It reads the PDF page by page and returns a Python list of `Document` objects. Each `Document` has two parts:
- `.page_content` -- the raw text on that page as a plain string
- `.metadata` -- a dictionary containing `"page"` (the page number, 0-indexed) and `"source"` (the file path)

One `Document` per page. A 9-page PDF gives 9 `Document` objects in the list.

In [ ]:
path = "../../data/RAI_Policy_Catalogue.pdf"

loader = PDFPlumberLoader(path)
raw_pages = loader.load()

print(f"Pages loaded: {len(raw_pages)}")
print("First 300 characters of page 1:")
print(raw_pages[0].page_content[:300])

---
## Step 3 -- Chunking

### Why do we chunk?

A full page of policy text is too long to match precisely in a vector search. When a proposal mentions "chatbot without disclosure," we want to retrieve the specific paragraph about chatbot transparency, not the entire page. Chunking cuts each page into smaller, more targeted pieces so the vector search returns the most relevant clause, not the most relevant page.

### What is RecursiveCharacterTextSplitter?

It tries to split text at the biggest natural break it can find, working through a priority list of separators:
1. `"\n\n"` -- blank lines (paragraph breaks) -- preferred, keeps ideas together
2. `"\n"` -- single newlines
3. `". "` -- sentence endings
4. `" "` -- spaces between words
5. `""` -- mid-word, as an absolute last resort

This is **format-agnostic**: it does not rely on any specific heading style or author convention. The moment a new author writes a clause differently, a regex-based splitter would silently break. `RecursiveCharacterTextSplitter` works regardless of who wrote the section.

Each chunk inherits the `.metadata` (including `"page"` number) from the page it came from. This is how we can label retrieved chunks with their source page in Tool 1.

### Our settings: chunk_size=600, chunk_overlap=80

- `chunk_size=600` -- each chunk targets roughly 600 characters (about 4-6 sentences)
- `chunk_overlap=80` -- each chunk repeats the last 80 characters of the previous chunk

The overlap stops a sentence from being cut in half at a boundary with the second half orphaned into the next chunk.

### Known tradeoff

600 characters is smaller than some policy sub-sections (Requirement, Violation Patterns). This means some sub-sections may be split across two chunks, slightly reducing retrieval precision. This was a deliberate tradeoff chosen for speed and simplicity. It can be improved later by increasing `chunk_size` to 800.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=80,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(raw_pages)
print(f"Total chunks created: {len(chunks)}")
print("\nFirst chunk text:")
print(chunks[0].page_content)

---
## Step 4 -- Embedding

### What is an embedding?

An embedding is a fixed-length list of numbers (a vector) that represents the *meaning* of a piece of text. Two chunks that mean similar things (both about "AI disclosure to users") will produce numerically similar vectors even if they use completely different words. This is what makes semantic search possible: we find relevant chunks by meaning, not by keyword matching.

### BAAI/bge-small-en-v1.5

This is a small, fast sentence embedding model from Hugging Face's BAAI (Beijing Academy of Artificial Intelligence). "Small" means it produces a 384-dimensional vector and runs on CPU without needing a GPU. "bge" stands for BAAI General Embedding. We chose this model because it is fast enough to run locally on a 6GB RAM machine with no GPU.

### model_kwargs={"device": "cpu"}

Explicitly tells the model to run on CPU, avoiding slow auto-detection on machines without a GPU. Without this, the model might try to find CUDA and produce a warning or delay.

### encode_kwargs={"normalize_embeddings": True}

Scales every output vector to length 1 (unit length, also called L2 normalisation). Chroma uses cosine similarity for its nearest-neighbour search, which measures the *angle* between two vectors, not their raw size. Normalising removes vector magnitude as a factor entirely, so similarity is judged purely on directional alignment -- which is what "does this text mean something similar" actually depends on.

### What does this object do?

`embeddings` is not yet computing anything at this point. It is a wrapper object that knows *how* to embed text. Chroma will call it internally in the next step when storing chunks and when searching.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print("Embedding model loaded.")

---
## Step 5 -- Vector Database (Chroma)

### What is a vector database?

A vector database stores text chunks as vectors and lets you search them by meaning. When we search for "chatbot not disclosing AI involvement," Chroma converts that query into a vector using the same embedding model, then returns the stored chunks whose vectors are closest to it by cosine similarity. The closer two vectors are, the more similar their meaning.

### Why a separate collection "rai_policies_v2"?

We already have a collection called `"rai_policies"` from earlier work (one chunk per page). This new collection uses the `RecursiveCharacterTextSplitter` with `chunk_size=600` (smaller, more precise chunks). Using a different collection name keeps the two separate inside the same Chroma `persist_directory` so they do not interfere with each other.

### persist_directory

Without `persist_directory`, Chroma only exists in memory and disappears when the kernel restarts. With it, the vectors are written to disk under the `"Chroma"` folder. On subsequent kernel restarts, the vectors are already there and do not need to be recomputed from scratch.

### IMPORTANT: delete_collection() before rebuilding

`Chroma.from_documents()` **appends** to an existing collection rather than replacing it. If you run this cell more than once without deleting first, you will get duplicate chunks (count = `len(chunks)` x number of runs). Always call `vector_db.delete_collection()` first if you need to rebuild.

The commented lines at the top of the code cell are there for exactly this purpose.

In [ ]:
# If you are running this for the first time, just run Chroma.from_documents directly.
# If you have run it before and want to rebuild cleanly, uncomment the two lines below first:
# vector_db = Chroma(collection_name="rai_policies_v2", embedding_function=embeddings, persist_directory="Chroma")
# vector_db.delete_collection()

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="rai_policies_v2",
    persist_directory="Chroma"
)

count = vector_db._collection.count()
print(f"Chunks stored in Chroma: {count}")
print(f"Match len(chunks)      : {count == len(chunks)}")

---
## Step 6 -- Jira Connection

### Why Jira?

Jira is where the project proposals live. Instead of hardcoding a proposal text in the notebook, we fetch real proposals from Jira issues. After the agent assesses a proposal, the findings are posted back to the same Jira issue as a comment. This closes the loop: the team sees the compliance findings directly in their project management tool.

### How Jira authentication works

Jira REST API v3 uses **Basic Authentication**: every request must include your email address and a personal API token encoded in the request header. The `HTTPBasicAuth` object handles this encoding automatically.

The `jira_auth` and `jira_headers` objects are created once here and reused by every Jira API call in the notebook. This avoids repeating the same setup in every function.

### Why REST API v3 and not the jira Python library?

We tried the `jira` Python library first, but it authenticates against an older API version (v2) that is no longer supported on new Atlassian accounts, causing persistent 401 errors. The `requests` library with the v3 endpoint works reliably on all account types.

### Why /rest/api/3/myself for the test?

This endpoint returns your own account information. If authentication is working, you get a 200 response with your display name and email. If anything is wrong with the credentials, you get a 401 immediately, before any real API call is made.

In [ ]:
JIRA_SERVER = os.environ.get("JIRA_SERVER")
JIRA_EMAIL  = os.environ.get("JIRA_EMAIL")
JIRA_TOKEN  = os.environ.get("JIRA_API_TOKEN")

jira_auth    = HTTPBasicAuth(JIRA_EMAIL, JIRA_TOKEN)
jira_headers = {"Accept": "application/json", "Content-Type": "application/json"}

# Test the connection -- must return status 200 before proceeding
response = requests.get(
    f"{JIRA_SERVER}/rest/api/3/myself",
    auth=jira_auth,
    headers=jira_headers
)
print("Status code :", response.status_code)
print("Connected as:", response.json()["displayName"])
print("Email       :", response.json()["emailAddress"])

---
## Step 7 -- Fetch Proposals from Jira

### Why fetch once at startup?

We call `fetch_proposals()` once here and store the results in `proposals`. Every subsequent step in the notebook reads from this list rather than making a new Jira API call. This means:
- The notebook runs faster (no repeated network requests)
- The proposal list is consistent throughout the session
- Your friend can call `fetch_proposals()` from Streamlit at app startup and reuse the same list

### What is JQL?

JQL stands for Jira Query Language. It works the same way as SQL but for Jira issues. The query `project = DE2 ORDER BY created ASC` means: "give me all issues in the DE2 project, ordered by creation date oldest first." `maxResults=50` limits the response to 50 issues.

### Why /rest/api/3/search/jql and not /rest/api/3/search?

The older `/rest/api/3/search` endpoint was deprecated by Atlassian in 2026. Using it returns a `410 Gone` error. The new endpoint is `/rest/api/3/search/jql`.

### What is Atlassian Document Format (ADF)?

Jira API v3 does not return description fields as plain text. It returns them as **Atlassian Document Format (ADF)**, a nested JSON structure where text is wrapped in content blocks and inline text nodes. The inner loop in `fetch_proposals()` walks through this nested structure and extracts just the plain text strings.

For example, a description like "Our chatbot handles queries automatically" is stored in Jira as:
```json
{"type": "doc", "content": [{"type": "paragraph", "content": [{"type": "text", "text": "Our chatbot handles queries automatically"}]}]}
```
The inner loop extracts just the `"text"` values from all `"content"` blocks.

In [ ]:
def fetch_proposals(project_key=None):
    if project_key is None:
        project_key = os.environ.get("JIRA_PROJECT_KEY")

    response = requests.get(
        f"{JIRA_SERVER}/rest/api/3/search/jql",
        auth=jira_auth,
        headers=jira_headers,
        params={
            "jql": f"project = {project_key} ORDER BY created ASC",
            "maxResults": 50,
            "fields": "summary,description"
        }
    )

    data = response.json()
    proposals = []
    for issue in data["issues"]:
        description_obj = issue["fields"].get("description")
        if description_obj is None:
            description_text = ""
        else:
            # Jira API v3 returns description as Atlassian Document Format (ADF)
            # ADF is a nested JSON structure: doc > paragraph > text
            # We walk through content blocks to extract plain text strings
            description_text = ""
            for block in description_obj.get("content", []):
                for inline in block.get("content", []):
                    if inline.get("type") == "text":
                        description_text += inline.get("text", "")
                description_text += "\n"

        proposals.append({
            "issue_key"  : issue["key"],
            "summary"    : issue["fields"]["summary"],
            "description": description_text.strip()
        })
    return proposals

# Fetch ALL proposals once at startup -- every step below reads from this list
proposals = fetch_proposals()
print(f"Fetched {len(proposals)} proposals from Jira.\n")
for i, p in enumerate(proposals):
    print(f"{i+1:>2}. {p['issue_key']}: {p['summary']}")

---
## Step 8 -- Tools

### What is a tool in the context of an LLM agent?

A tool is a Python function that the LLM can call by name during reasoning. The LLM does not run the code itself. It decides "I need to call this function with these arguments" and LangGraph executes it and sends the result back to the LLM as a `ToolMessage` appended to the conversation history.

### What is the @tool decorator?

The `@tool` decorator from `langchain_core.tools` transforms a plain Python function into a `BaseTool` object. It does three things:
1. Reads the function's **docstring** and sets it as the tool's description -- this is the only description the LLM gets, so it must be precise
2. Inspects the **type hints** (`text:str`, `k:int=3`) and builds a JSON schema for the arguments -- this is how the LLM knows what to send
3. Wraps the return value in a `ToolMessage` so LangGraph can route it back into the conversation

### Why do tools return strings?

Tools must return simple text because the LLM can only read text, not Python objects like DataFrames, dictionaries, or lists. Every tool formats its output as a readable string before returning it.

### Why 6 tools instead of 5?

The original notebook had 5 tools covering RAG retrieval, XAI grounding, severity lookup, audit logging, and structured verdict. We added a 6th tool, `llm_assessment`, which produces a free-form narrative assessment in plain language. This gives the human reviewer and your friend's Streamlit dashboard a readable summary alongside the structured findings.

### Our 6 tools

| Tool | Role | What the LLM uses to call it |
|------|------|------------------------------|
| `search_policies` | RAG retrieval from Chroma | A query based on trigger words |
| `get_trigger_words` | XAI: which words drove the classifier | The full proposal text |
| `get_policy_severity` | Fixed lookup | The policy ID found in search results |
| `log_decision` | Article 12 audit trail | Proposal text, prediction, probability |
| `write_compliance_verdict` | Structured findings | Policy ID and reason |
| `llm_assessment` | Plain language narrative | The full proposal text |

### Tool 1 -- search_policies (RAG Retrieval)

This tool is the **RAG (Retrieval Augmented Generation)** component of the agent.

**What is RAG?**
RAG means: instead of asking the LLM to remember policy text from its training data, we retrieve the relevant text at runtime and give it fresh as context. This is more accurate and more auditable than relying on LLM memory, because we can always trace exactly which policy text the LLM read before making a decision.

**How it works:**
The LLM calls this tool with a short query string based on the trigger words it found in Tool 2. Chroma converts the query into a vector using the same embedding model used in Step 4, then returns the `k` stored chunks whose vectors are closest (most similar in meaning) to the query vector.

**Deduplication with `seen`:**
Because `chunk_size=600` is small relative to the policy text, the same sentence can appear in multiple overlapping chunks. Without deduplication, the LLM would read the same policy clause 2-3 times and waste reasoning capacity on it. We use a Python `set` called `seen` to track which chunks have already been added. Before adding a chunk, we take its first 100 characters as a "fingerprint" and check if that fingerprint is already in `seen`. If it is, we skip that chunk with `continue`. A `set` only stores unique values, so adding the same fingerprint twice has no effect.

In [ ]:
@tool
def search_policies(query:str, k:int=3)->str:
    """
    Search the RAI policy catalogue for clauses relevant to a query.
    Use this to find which of the 9 RAI policies apply to a proposal.
    Args:
        query: a short description of the concern or topic to search for
        k: number of matching policy chunks to return
    Returns:
        the matched policy text chunks, each labelled with its source page
    """
    results = vector_db.similarity_search(query, k=k)

    output = ""
    seen = set()
    for doc in results:
        page = doc.metadata.get("page", "unknown")
        fingerprint = doc.page_content[:100].strip()
        if fingerprint in seen:
            continue
        seen.add(fingerprint)
        output += f"[page {page}]\n{doc.page_content}\n\n"
    return output

# Sanity check
# print(search_policies.invoke({"query": "chatbot not disclosing it is an AI", "k": 2}))

### Tool 2 -- get_trigger_words (XAI Grounding)

This tool exposes the **explainability layer from Goal 3** as something the LLM can call itself.

**What are trigger words?**
The Logistic Regression model assigns a weight (`coef_[0]`) to every word in the vocabulary. A positive weight means the word pushes the prediction toward Red Flag. A negative weight means it pushes toward Compliant.

For any specific proposal, the contribution of each word is:
`contribution = tfidf_score x weight`

Where `tfidf_score` is how prominently that word appears in this specific proposal. Words that do not appear in the proposal have `tfidf_score = 0`, so their contribution is exactly 0. This is what makes this a **per-proposal** explanation rather than a global model summary: only words actually in this proposal count.

**Why does this matter for the LLM?**
Without trigger words, the LLM would have to guess why the classifier flagged this proposal. With trigger words, the LLM can say "the classifier flagged this because of words like `automatically`, `without`, `users` -- these are transparency-related terms." This grounds the LLM's reasoning in real classifier evidence, not guesswork.

**EU AI Act Article 14:**
Article 14 requires that a human can effectively oversee the system's operation, including understanding why a specific output was produced. Trigger words satisfy this requirement: the reviewer sees exactly which words in the proposal caused the classifier to flag it.

**Where `explain_prediction` comes from:**
This function is defined in `xai2.py` and was imported in Step 2a. It iterates through the vocabulary, skips absent words, computes `tfidf_score x weight` for each present word, sorts by absolute contribution descending, and returns the top `n` as a list of `(word, score)` tuples.

In [ ]:
@tool
def get_trigger_words(text:str, n:int=10)->str:
    """
    Find which words in this specific proposal drove the classifier's prediction.
    Use this to ground your reasoning in the model's actual evidence for this proposal.
    Args:
        text: the proposal text to explain
        n: how many top contributing words to return
    Returns:
        the top words with their contribution scores and direction
    """
    results = explain_prediction(classifier_pipeline, text, n)
    output = ""
    for word, score in results:
        direction = "toward Red Flag" if score > 0 else "toward Compliant"
        output += f"{word}: {round(score, 4)} ({direction})\n"
    return output

# Sanity check
# print(get_trigger_words.invoke({
#     "text": "Our chatbot will handle customer complaints automatically without telling users it is an AI.",
#     "n": 5
# }))

### Tool 3 -- get_policy_severity (Fixed Lookup)

Severity is **NOT** something the LLM should reason about or decide. It is a fixed value defined in the RAI Policy Catalogue itself. Policies RAI-01 through RAI-05 are always High severity. Policies RAI-06 through RAI-09 are always Medium severity. These values never change based on context.

**Why a lookup table instead of LLM reasoning?**
If we let the LLM decide severity, it might give different answers on different runs due to its stochastic nature. A deterministic lookup table guarantees the same answer every single time for the same policy ID. This is important for two reasons:
1. **Routing consistency**: the `human_gate` node reads `state["severity"]` to decide whether to pause for human input. An inconsistent severity value would produce inconsistent routing.
2. **Article 12 reproducibility**: the audit log must record a severity that can be reproduced from the same input.

**Why .upper() on the policy_id?**
The LLM might pass `"rai-02"` or `"Rai-02"` instead of `"RAI-02"`. `.upper()` converts whatever case the LLM uses to uppercase before the lookup, so all variations work correctly.

**Why .get(key, None) instead of [key]?**
`.get(key, None)` returns `None` if the key does not exist, instead of crashing with a `KeyError`. This lets us handle the case where the LLM passes a made-up or incorrectly formatted policy ID gracefully, returning a helpful error message instead of breaking the agent loop.

**Why severity matters for routing:**
- High severity (RAI-01 to RAI-05) routes to the `human_gate` node where a human must type APPROVE or REJECT
- Medium severity (RAI-06 to RAI-09) auto-approves in `human_gate` with no human input required

This is the Human-in-the-Loop gate required by EU AI Act Article 14.

In [ ]:
@tool
def get_policy_severity(policy_id:str)->str:
    """
    Look up the fixed severity level for a given RAI policy.
    Severity is a fixed value from the policy catalogue, not something to reason about.
    Use this to determine whether a violation needs human approval (High) or can proceed automatically (Medium).
    Args:
        policy_id: the policy identifier, e.g. RAI-01, RAI-02, up to RAI-09
    Returns:
        the severity level, either High or Medium, and what it means for routing
    """
    severity_lookup = {
        "RAI-01": "High", "RAI-02": "High", "RAI-03": "High",
        "RAI-04": "High", "RAI-05": "High", "RAI-06": "Medium",
        "RAI-07": "Medium", "RAI-08": "Medium", "RAI-09": "Medium",
    }
    severity = severity_lookup.get(policy_id.upper(), None)
    if severity is None:
        return f"{policy_id} is not a recognised RAI policy. Valid options are RAI-01 through RAI-09."
    if severity == "High":
        return f"{policy_id} is High severity. This violation must be routed to a human reviewer for approval before any action is taken."
    return f"{policy_id} is Medium severity. This violation may proceed to automatic remediation with an optional spot check."

# Sanity check
# print(get_policy_severity.invoke({"policy_id": "RAI-02"}))
# print(get_policy_severity.invoke({"policy_id": "RAI-07"}))
# print(get_policy_severity.invoke({"policy_id": "RAI-99"}))

### Tool 4 -- log_decision (EU AI Act Article 12 Audit Trail)

Every compliance decision must be logged in a way that allows full reconstruction after the fact. This is required by **EU AI Act Article 12**, which states that high-risk AI systems must automatically log events throughout their operation in a way that enables traceability of the system's functioning.

**What gets logged:**
- `actual` -- the true label (1 = Red Flag, 0 = Compliant)
- `predicted` -- what the model predicted
- `probability` -- how confident the model was (e.g. 0.6427)
- `correct` -- whether the prediction matched the true label
- `trigger_words` -- which words in this specific proposal drove the prediction
- `explanation_method` -- always `"LogReg coef_ x tfidf"` -- hardcoded, never changes

**Why is `explanation_method` hardcoded?**
LogReg coefficients are exact and deterministic. The same input always produces the same `coef_` values and the same `tfidf_score` values, which means the same contribution scores, which means the same trigger words, which means the same explanation. This is not an approximation. An auditor reading this log entry two years from now can reproduce the exact same explanation from the saved model without any additional context.

This is what makes it legally defensible under Article 12. If we used XGBoost with SHAP approximations instead, the explanation could vary slightly between runs and would not be fully reproducible.

**Where `audit_trail` comes from:**
`audit_trail()` is defined in `xai2.py` and was imported in Step 2a. It calls `explain_prediction()` internally to get the trigger words, then builds the log dictionary. `log_decision` wraps it and formats the output as a readable string for the LLM.

In [ ]:
@tool
def log_decision(text:str, y_true:int, y_pred:int, proba:float, n_terms:int=10)->str:
    """
    Log a compliance decision for audit purposes.
    Use this as your final action once you have reached a conclusion about a proposal.
    Satisfies EU AI Act Article 12: every decision is logged with the exact inputs,
    prediction, probability, and trigger words that produced it.
    Args:
        text: the proposal text that was assessed
        y_true: the true label, 1 for Red Flag, 0 for Compliant
        y_pred: the predicted label, 1 for Red Flag, 0 for Compliant
        proba: the classifier's predicted probability of Red Flag
        n_terms: how many trigger words to include in the log
    Returns:
        a formatted string of the full audit log entry
    """
    log = audit_trail(classifier_pipeline, text, y_true, y_pred, proba, n_terms)

    output = ""
    output += f"Actual label      : {'Red Flag' if log['actual']==1 else 'Compliant'}\n"
    output += f"Predicted label   : {'Red Flag' if log['predicted']==1 else 'Compliant'}\n"
    output += f"Probability       : {log['probability']}\n"
    output += f"Correct           : {log['correct']}\n"
    output += f"Trigger words     : {', '.join(log['trigger_words'])}\n"
    output += f"Explanation method: {log['explanation_method']}\n"
    return output

# Sanity check
print(log_decision.invoke({
    "text": "Our chatbot will handle customer complaints automatically without telling users it is an AI.",
    "y_true": 1,
    "y_pred": 1,
    "proba": 0.87,
    "n_terms": 5
}))

### Tool 5 -- write_compliance_verdict (Structured Findings)

This is the **last structured tool** the LLM calls after `log_decision`. It produces a structured, human-readable compliance finding that answers two questions:
1. **Which policy was violated?** -- grounded in what `search_policies` returned
2. **Why was it flagged?** -- grounded in the trigger words from `get_trigger_words`

**Where do the arguments come from?**
- `policy_id` -- the LLM determined this from reading the chunks returned by `search_policies` and the severity confirmed by `get_policy_severity`. The LLM does not invent it; it reads it from the tool results it already received.
- `reason` -- the LLM writes this itself, combining what it saw in `get_trigger_words` (the words that drove the flag) with what it read in `search_policies` (the policy requirement that was not met). This is where the LLM's reasoning happens.

**Why is severity looked up deterministically inside this tool again?**
The verdict must include severity for the human reviewer to understand the urgency. We look it up deterministically here (same lookup table as Tool 3) rather than passing it as an argument from the LLM, because we never want the LLM to decide or pass severity values -- only to look them up from the fixed table.

**Why is there no `recommended_fix` argument?**
The human reviewer decides the fix. The agent's job is to identify the problem clearly and explain it. Generating a fix automatically would overstep the Human-in-the-Loop design and bypass the review that EU AI Act Article 14 requires.

In [ ]:
@tool
def write_compliance_verdict(policy_id:str, reason:str)->str:
    """
    Write the final compliance verdict for a flagged proposal.
    Call this as your last structured action after log_decision.
    This identifies the policy violated and explains the reason.
    It does NOT suggest a fix -- that is for the human reviewer to decide.
    Args:
        policy_id: the policy that was violated, e.g. RAI-02
        reason: one sentence explaining what the proposal is missing,
                grounded in the trigger words and retrieved policy text
    Returns:
        a formatted compliance verdict string
    """
    severity_lookup = {
        "RAI-01": "High", "RAI-02": "High", "RAI-03": "High",
        "RAI-04": "High", "RAI-05": "High", "RAI-06": "Medium",
        "RAI-07": "Medium", "RAI-08": "Medium", "RAI-09": "Medium",
    }
    severity = severity_lookup.get(policy_id.upper(), "Unknown")

    output = ""
    output += f"POLICY VIOLATED : {policy_id.upper()}\n"
    output += f"SEVERITY        : {severity}\n"
    output += f"REASON          : {reason}\n"
    output += f"FIX             : PENDING HUMAN REVIEW\n"
    return output

### Tool 6 -- llm_assessment (Free-Form Narrative)

This is the **new tool** added in this version. Unlike the other 5 tools which produce structured, deterministic outputs, this tool asks the LLM to reason freely about the proposal in plain language.

**Why add a narrative tool?**
The structured verdict (Tool 5) answers: what policy, what severity, what reason. The narrative answers: what is actually going on with this proposal in human terms? It is the difference between a legal finding and an explanation you would give a colleague. The reviewer needs both.

**How this tool differs from the others:**
- It makes its own separate LLM call inside the tool function, with `temperature=0.3` (slightly creative) to produce natural, readable language
- The main agent LLM uses `temperature=0.0` for determinism; this internal call uses 0.3 for readability
- It receives the full proposal text and writes a 3-5 sentence assessment in plain English

**What goes in the narrative:**
- What the proposal is trying to do
- What the main compliance concern is
- How serious the concern is in practical terms

**What does NOT go in the narrative:**
The system prompt inside this tool explicitly says "Do NOT suggest a fix." The fix is for the human reviewer to decide after reading both the structured verdict and the narrative.

**Streamlit integration:**
Your friend can display this narrative as the "AI Commentary" section on the dashboard, giving reviewers a plain English explanation alongside the structured findings.

In [ ]:
@tool
def llm_assessment(proposal_text:str)->str:
    """
    Generate a free-form plain language assessment of this proposal.
    Call this after gathering evidence from other tools to provide a human-readable
    narrative explanation of your findings and any nuances the structured verdict cannot capture.
    Args:
        proposal_text: the full proposal text to assess
    Returns:
        a plain language narrative assessment of the proposal in 3-5 sentences
    """
    # This tool makes its own internal LLM call with temperature=0.3 for natural language
    # The main agent uses temperature=0.0 for determinism
    # This separate call is what allows the narrative to read naturally
    assessment_llm = ChatGroq(
        model="llama-3.1-8b-instant",
        temperature=0.3, # If you prefer consistency over readability, change it to 0.0
        api_key=os.environ.get("GROQ_API_KEY")
    )
    system = SystemMessage(content="""You are a Responsible AI compliance expert reviewing internal project proposals.
Write a clear plain language assessment for a human reviewer.
Cover: what the proposal is trying to do, what the main compliance concern is, how serious it is in practical terms.
Write in 3-5 sentences. Be direct and practical.
Do NOT suggest a fix -- that is for the human reviewer to decide.""")
    human = HumanMessage(content=f"Proposal to assess:\n\n{proposal_text}")
    response = assessment_llm.invoke([system, human])
    return response.content

---
## Step 9 -- LLM Setup

### Why Groq instead of a local model?

We originally planned to use `phi4-mini` (2.5GB) running locally through Ollama. During testing, phi4-mini could not produce structured tool calls reliably. It wrote tool names as plain text inside the message `content` field instead of in the `tool_calls` field. LangGraph's `ToolNode` only reads `tool_calls`, so no tools ever executed.

This is a model capability threshold: models below roughly 7-8 billion parameters typically cannot reliably produce the JSON-structured tool call format that LangChain requires.

We switched to Groq's free API running `llama-3.1-8b-instant`. Groq runs the model on their servers so our local 6GB RAM is not a constraint. `llama-3.1-8b-instant` handles structured tool calling correctly.

### temperature=0.0

Temperature controls how creative vs deterministic the LLM is. `0.0` means fully deterministic: the same input always produces the same output. This is essential for a compliance system where reproducibility is required by EU AI Act Article 12. Note: Tool 6 (`llm_assessment`) uses its own separate LLM call with `temperature=0.3` to produce more natural narrative language.

### bind_tools(tools)

`llm.bind_tools(tools)` attaches all 6 tools to the LLM permanently. It sends each tool's name, docstring, and argument schema to Groq alongside every request. After this, the LLM can see all 6 tools and decide on its own which ones to call, in what order, and with what arguments. The LLM is in control of the reasoning loop.

### Security note

Your API key is loaded from `.env` via `os.environ.get("GROQ_API_KEY")`. It is never visible in the notebook code. Before sharing this notebook, confirm that your `.env` file is listed in `.gitignore` and has never been committed to Git.

In [ ]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.0,
    api_key=os.environ.get("GROQ_API_KEY")
)

tools = [
    search_policies,
    get_trigger_words,
    get_policy_severity,
    log_decision,
    write_compliance_verdict,
    llm_assessment,
]

llm_with_tools = llm.bind_tools(tools)
print("LLM loaded and tools bound.")
print(f"Tools available to LLM: {[t.name for t in tools]}")

---
## Step 10 -- LangGraph Agent

### What is LangGraph?

LangGraph is a library for building AI agents as graphs. A graph has two things: **nodes** (steps that do work) and **edges** (connections between steps). Think of it like a flowchart. Each box in the flowchart is a node. Each arrow between boxes is an edge. LangGraph runs the graph by following the arrows from one node to the next.

### Our graph has 4 nodes

| Node | What it does | When it runs |
|------|-------------|--------------|
| `classify` | Runs Signal 1 (the ML classifier) | Always first, every proposal |
| `agent` | Calls the LLM with all 6 tools | Only when classifier said Red Flag |
| `tools` | Executes whichever tool the LLM requested | Every time LLM requests a tool |
| `human_gate` | Pauses for human approval or auto-approves | After LLM finishes reasoning |

### Our graph has these edges

```
START --> classify                          (fixed edge: always start at classify)
classify --> agent       (if y_pred == 1)  (conditional: Red Flag goes to agent)
classify --> END         (if y_pred == 0)  (conditional: Compliant stops here)
agent --> tools          (if tool_calls non-empty)  (conditional: LLM wants a tool)
agent --> human_gate     (if tool_calls empty)       (conditional: LLM is done)
tools --> agent                            (fixed edge: always loop back to agent)
human_gate --> END                         (fixed edge: always stop after decision)
```

**Fixed edges** always go to the same next node regardless of state. **Conditional edges** call a routing function to decide where to go next.

### The tool-call loop

The `tools --> agent` fixed edge creates a loop. Every time a tool executes, the graph goes back to `agent` so the LLM can read the tool result and decide what to do next. This continues until the LLM produces a response with no tool calls, at which point the graph exits the loop and goes to `human_gate`.

### The GATE_MODE flag

The `human_gate` node reads a global `GATE_MODE` variable at runtime:
- `"interactive"` -- pauses and waits for you to type APPROVE or REJECT (for notebook use)
- `"auto"` -- auto-approves without pausing (for Streamlit or scripted use)

We compile the graph **once**. Changing `GATE_MODE` before running changes the behaviour without rebuilding the graph.

### Step 10a -- AgentState (Shared Memory)

AgentState is the shared memory of the graph. Every node reads from it and writes back to it. It is defined as a `TypedDict` (a Python dictionary with fixed key names and declared types). LangGraph creates one instance at the start of a run and passes it to every node in turn.

Think of it as a form that gets filled in progressively as the graph runs. `classify` fills in `y_pred` and `proba`. The LLM (via `get_policy_severity`) fills in `severity`. `human_gate` fills in `human_approved` and `final_decision`.

**The special field: `messages: Annotated[list, operator.add]`**

`Annotated[list, operator.add]` is a LangGraph-specific pattern. `operator.add` is the `+` operator expressed as a function. This metadata tells LangGraph: when a node returns new messages, **append** them to the existing list rather than overwriting it. Without this, every node call would replace the entire conversation history and the LLM would forget all previous tool results on every step.

In [ ]:
class AgentState(TypedDict):
    # The raw proposal text being assessed. Set at the start, never changes.
    proposal_text: str
    # The classifier's hard prediction. 1 = Red Flag, 0 = Compliant.
    y_pred: int
    # The classifier's Red Flag probability score. e.g. 0.6427
    proba: float
    # High or Medium. Written by the LLM after calling get_policy_severity.
    severity: str
    # Full conversation history. Annotated means messages are APPENDED not overwritten.
    messages: Annotated[list, operator.add]
    # True if human approved or Medium severity auto-approved. False if rejected.
    human_approved: bool
    # Plain English summary of the final outcome.
    final_decision: str

print("AgentState defined.")

### Step 10b -- Node 1: classify

This node runs **Signal 1**: the ML classifier. It always runs first, on every proposal, regardless of what the proposal says. It is fast (under 1 second) and never calls the LLM or any tool.

**What it does:**
1. Reads `proposal_text` from `AgentState`
2. Calls `classifier_pipeline.predict_proba([text])` to get the Red Flag probability
3. Calls `classifier_pipeline.predict([text])` to get the hard 0/1 decision (threshold 0.4 applied internally by `ThresholdAdjustor`)
4. Returns `y_pred` and `proba` to be merged into `AgentState`

**Why nodes return partial dicts:**
Nodes never return the full `AgentState`. They return only the fields they changed. LangGraph automatically merges the returned dict back into the full state. This is why `classify` only returns `{"y_pred": ..., "proba": ...}` and not all 7 fields.

**What happens next:**
`route_after_classify` reads `y_pred`. If 1 (Red Flag), the graph goes to `agent`. If 0 (Compliant), the graph stops at `END` immediately -- no LLM call, no tools, nothing.

In [ ]:
# Global gate mode flag -- change this before running to switch behaviour
# "interactive" -- pauses for human input on High severity (for notebook use)
# "auto"        -- auto-approves on High severity (for Streamlit or scripted use)
GATE_MODE = "interactive"

def classify(state:AgentState)->dict:
    # Read the proposal text from shared state
    text = state["proposal_text"]

    # predict_proba returns shape (1, 2): [[prob_compliant, prob_red_flag]]
    # [0][1] gets the Red Flag probability for the first (only) document
    proba = classifier_pipeline.predict_proba([text])[0][1]

    # predict() applies the threshold (0.4) internally via ThresholdAdjustor
    # Returns 1 for Red Flag, 0 for Compliant
    # int() converts from numpy.int64 to plain Python int
    y_pred = int(classifier_pipeline.predict([text])[0])

    # Return only the fields this node changes.
    # LangGraph merges this partial dict into AgentState automatically.
    return {
        "y_pred": y_pred,
        "proba": round(float(proba), 4),
    }

print("classify node defined.")

### Step 10b -- Node 2: agent

This node calls the LLM with all 6 tools available. It runs in a loop with the `tools` node until the LLM decides it is done reasoning.

**The system prompt:**
The system prompt is deliberately thin. It tells the LLM what tools exist and what its job is, then steps back. It does not prescribe which tool to call first, in what order, or how many times. The LLM reads the proposal from the conversation history (`state["messages"]`) and decides on its own how to use the tools to reach a conclusion.

This is what makes the system a **genuine agent** rather than a scripted pipeline. The LLM may call `search_policies` twice with different queries if the first result was not specific enough. It may check trigger words before or after searching policies. It controls the reasoning loop entirely.

**Why {{double braces}} around tool names in the prompt?**
Inside an f-string, single `{}` means "evaluate this Python expression." We want the LLM to see literal `{get_trigger_words}` as a visual marker identifying exact tool names, not have Python try to evaluate `get_trigger_words` as a variable. Double braces `{{get_trigger_words}}` produce a literal `{get_trigger_words}` in the output string.

**The tool-call loop:**
After each LLM response, `route_after_agent` checks the last message. If it has non-empty `tool_calls`, `ToolNode` runs the requested tool and sends the result back to this node. The LLM reads the result and decides what to do next. This continues until the LLM produces a response with empty `tool_calls`, at which point the graph moves to `human_gate`.

**Known tradeoff:**
Full autonomy requires a model capable of multi-step reasoning without explicit step-by-step guidance. On a small model like `llama-3.1-8b-instant`, a more scripted prompt can be more reliable. The autonomous version is architecturally correct and satisfies the design requirement, but may behave inconsistently at this model size. A larger model handles full autonomy without issue.

In [ ]:
def agent(state:AgentState)->dict:
    system=SystemMessage(content="""You are a compliance assessment agent for a Responsible AI policy system.

You have access to these tools:
- {get_trigger_words}: find which words in the proposal drove the classifier prediction
- {search_policies}: search the RAI policy catalogue for relevant clauses
- {get_policy_severity}: look up the severity of a specific RAI policy
- {log_decision}: write the audit trail entry for this decision
- {write_compliance_verdict}: write the final structured compliance verdict (policy and reason only, NO fix)
- {llm_assessment}: generate a free-form plain language assessment of the proposal

Use your tools in whatever order and however many times you judge necessary to accurately assess whether this proposal violates any RAI policy.
When you are confident in your assessment, call {log_decision} then {write_compliance_verdict} to conclude.
IMPORTANT: Do NOT suggest a recommended fix in the verdict. That is for the human reviewer to decide.""")

    messages=[system]+state["messages"]
    response=llm_with_tools.invoke(messages)
    return{"messages":[response]}

### Step 10b -- Node 3: human_gate

This node implements the **Human-in-the-Loop gate** required by EU AI Act Article 14. Article 14 requires that a natural person can effectively oversee the operation of a high-risk AI system, including the ability to understand the output, decide not to use it, and intervene before it takes effect.

**How GATE_MODE works:**
The node reads the global `GATE_MODE` variable set at the top of the nodes cell:
- `"interactive"` (default for notebook use): prints the full agent reasoning and pauses for human input
- `"auto"` (for Streamlit or scripted use): auto-approves without pausing, flags the result as pending dashboard review

This means we compile the graph **once** and change the mode simply by setting `GATE_MODE = "auto"` before calling `assess_proposal()`. No graph rebuilding needed.

**Medium severity:**
Auto-approved immediately. The system proceeds without human input. Medium severity violations (RAI-06 to RAI-09) are lower risk and do not require blocking human review.

**High severity:**
The node prints the full conversation history, showing every tool the LLM called and every result it received. This is the evidence the human reviewer uses to make their decision. The reviewer then types APPROVE or REJECT.

**Why `input()` works here:**
`input()` is a built-in Python function that pauses all Python execution and waits for keyboard input. `MemorySaver` has already saved the full graph state to memory before this pause, so the graph can resume from exactly this point when the human responds.

**What the human reviewer sees:**
In interactive mode, the reviewer sees the full message history filtered to show only meaningful content: tool results (labelled by tool name) and plain LLM text messages. Raw tool call requests (which have empty `content` and only `tool_calls`) are filtered out because they have no human-readable value.

In [ ]:
def human_gate(state:AgentState)->dict:
    severity=state["severity"]

    if severity=="Medium":
        print("Medium severity: proceeding automatically.")
        return{
            "human_approved":True,
            "final_decision":"Medium severity violation. Automatic remediation approved."
        }

    if GATE_MODE == "auto":
        # Auto mode: used by Streamlit or scripted calls -- no input() blocking
        # High severity is still flagged clearly so the dashboard can handle it
        return{
            "human_approved":True,
            "final_decision":"High severity violation. Auto-approved. PENDING HUMAN REVIEW on dashboard."
        }

    # Interactive mode: pause for human input
    print("\n--- HIGH SEVERITY: HUMAN APPROVAL REQUIRED ---")
    print(f"Proposal text:\n{state['proposal_text']}\n")
    print("Agent reasoning and tool outputs:")
    for message in state["messages"]:
        if hasattr(message, "name") and message.name and message.content:
            # Tool result messages have .name set to the tool function name
            print(f"[{message.name}] {message.content}")
        elif message.content and not hasattr(message, "tool_calls"):
            # Plain LLM text messages (not tool call requests)
            print(f"[LLM] {message.content}")
    print("----------------------------------------------")

    decision=input("Type APPROVE or REJECT: ").strip().upper()

    if decision=="APPROVE":
        return{
            "human_approved":True,
            "final_decision":"High severity violation. Human reviewer approved. Pending fix decision."
        }

    return{
        "human_approved":False,
        "final_decision":"High severity violation. Human reviewer rejected. No action taken."
    }

print("Nodes defined.")

### Step 10c -- Routing Functions

Routing functions are **not nodes**. They are plain Python functions that LangGraph calls after a node finishes to decide which node runs next. They receive the current `state` and return a string: the name of the next node, or the constant `END` to stop the graph.

**route_after_classify:**
Reads `state["y_pred"]`. If 1 (Red Flag), returns `"agent"` -- the graph continues to Signal 2. If 0 (Compliant), returns `END` -- the graph stops immediately with no LLM call.

**route_after_agent:**
Reads `state["messages"][-1]`, which is the last message the LLM just produced. `[-1]` in Python always means the last item in a list.

- `hasattr(last_message, "tool_calls")` -- `hasattr` is a built-in Python function that returns `True` if the object has the named attribute. Not all LangChain message types have a `tool_calls` attribute, so we check before reading it.
- `len(last_message.tool_calls) > 0` -- confirms the list is non-empty. The LLM might produce a message with `tool_calls` as an empty list if it decided not to call anything, so we check the length too.
- If both are true: LLM wants a tool, go to `"tools"`
- Otherwise: LLM is done reasoning, go to `"human_gate"`

In [ ]:
def route_after_classify(state:AgentState)->str:
    if state["y_pred"] == 1:
        return "agent"
    return END

def route_after_agent(state:AgentState)->str:
    last_message = state["messages"][-1]
    # hasattr: built-in that returns True if the object has that attribute
    # len(...) > 0: confirms LLM actually requested a tool (not empty list)
    if hasattr(last_message, "tool_calls") and len(last_message.tool_calls) > 0:
        return "tools"
    return "human_gate"

print("Routing functions defined.")

### Step 10d -- Build and Compile the Graph (done ONCE)

The graph is built and compiled **exactly once** here. We never rebuild it. Switching between interactive and auto mode is done by changing the `GATE_MODE` global variable, not by recompiling.

**Step-by-step walkthrough of the build:**

1. `StateGraph(AgentState)` -- creates an empty graph using `AgentState` as its shared memory structure
2. `add_node(name, function)` -- registers each Python function as a named node. The string name is what routing functions return and what edges reference
3. `ToolNode(tools)` -- a prebuilt LangGraph node. It reads the `tool_calls` field from the last LLM message, finds the matching tool in the `tools` list, calls it with the LLM's arguments, and appends the result as a `ToolMessage` to `state["messages"]`
4. `add_edge(START, "classify")` -- fixed edge: the graph always starts at `classify`
5. `add_conditional_edges("classify", route_after_classify, {...})` -- after `classify`, call `route_after_classify(state)` and look up the return value in the dict to find the next node
6. `add_conditional_edges("agent", route_after_agent, {...})` -- same pattern for the agent routing
7. `add_edge("tools", "agent")` -- fixed edge: after any tool runs, always go back to `agent`. This is the tool-call loop.
8. `add_edge("human_gate", END)` -- fixed edge: after the human decision, always stop
9. `MemorySaver()` -- an in-memory checkpoint store that saves the full `AgentState` after every node runs. This is what allows `input()` in `human_gate` to pause the graph safely
10. `graph_builder.compile(checkpointer=memory)` -- finalises the graph into a runnable `CompiledStateGraph` object. After this, no more nodes or edges can be added.

In [ ]:
# Step 1: Create an empty graph with our state definition
graph_builder = StateGraph(AgentState)

# Step 2: Register nodes
graph_builder.add_node("classify", classify)
graph_builder.add_node("agent", agent)
graph_builder.add_node("tools", ToolNode(tools))
graph_builder.add_node("human_gate", human_gate)

# Step 3: Add edges
# Fixed edge: always start at classify
graph_builder.add_edge(START, "classify")

# Conditional edge: after classify, route based on y_pred
# route_after_classify returns "agent" (Red Flag) or END (Compliant)
graph_builder.add_conditional_edges(
    "classify",
    route_after_classify,
    {"agent": "agent", END: END}
)

# Conditional edge: after agent, route based on whether LLM called a tool
# route_after_agent returns "tools" (tool requested) or "human_gate" (LLM done)
graph_builder.add_conditional_edges(
    "agent",
    route_after_agent,
    {"tools": "tools", "human_gate": "human_gate"}
)

# Fixed edge: after any tool executes, always loop back to agent
# This is the tool-call loop: agent -> tools -> agent -> tools -> ...
graph_builder.add_edge("tools", "agent")

# Fixed edge: after human_gate, always stop
graph_builder.add_edge("human_gate", END)

# Step 4: Compile with MemorySaver so state persists across the human gate pause
memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

print("Graph compiled successfully. This runs only once.")

---
## Step 11 -- Jira Write-Back Helpers

After the agent assesses a proposal, we post the findings back to the same Jira issue as a comment. This closes the loop: the team sees the compliance findings directly in their project management tool without needing to check a separate dashboard.

### _build_adf_comment(text)

Jira REST API v3 does not accept plain text in comment bodies. It requires **Atlassian Document Format (ADF)**, the same nested JSON structure used for descriptions. This helper converts a plain text string into the correct ADF structure.

Each non-empty line of text becomes one ADF paragraph node containing one text node:
```json
{"type": "paragraph", "content": [{"type": "text", "text": "your line here"}]}
```
Multiple paragraphs are wrapped in a top-level `doc` node with `version: 1`.

### post_jira_comment(issue_key, comment_text)

Posts the ADF-formatted comment to a specific Jira issue using the REST API v3 comments endpoint: `/rest/api/3/issue/{issue_key}/comment`.

A successful response returns HTTP status `201 Created`. Any other status means the comment was not posted, and we print the error details for debugging.

In [ ]:
def _build_adf_comment(text:str)->dict:
    # Jira API v3 requires comments in Atlassian Document Format (ADF)
    # Each non-empty line becomes a paragraph node containing a text node
    paragraphs = []
    for line in text.split("\n"):
        if line.strip():
            paragraphs.append({
                "type": "paragraph",
                "content": [{"type": "text", "text": line}]
            })
    return {
        "body": {
            "version": 1,
            "type": "doc",
            "content": paragraphs or [
                {"type": "paragraph", "content": [{"type": "text", "text": text}]}
            ]
        }
    }

def post_jira_comment(issue_key:str, comment_text:str)->bool:
    # Posts a comment on a Jira issue
    # Returns True if successful (HTTP 201), False otherwise
    url = f"{JIRA_SERVER}/rest/api/3/issue/{issue_key}/comment"
    response = requests.post(
        url,
        auth=jira_auth,
        headers=jira_headers,
        json=_build_adf_comment(comment_text)
    )
    if response.status_code == 201:
        print(f"  Comment posted to {issue_key} successfully.")
        return True
    print(f"  Failed to post to {issue_key}. Status: {response.status_code}")
    print(f"  Response: {response.text[:200]}")
    return False

print("Jira write-back helpers defined.")

---
## Step 12 -- Core Assessment Function

`assess_proposal(p)` is the single function that runs one proposal through the complete pipeline:

1. **Signal 1**: runs the classifier, gets `y_pred` and `proba`
2. **If Compliant** (`y_pred == 0`): builds a Compliant comment and posts it to Jira. Done.
3. **If Red Flag** (`y_pred == 1`): runs Signal 2 (the LLM agent with all 6 tools)
4. **Extracts results** from the message history by searching backwards for specific tool result messages
5. **Posts findings** to Jira as a structured comment
6. **Saves the result** to `compliance_results.json` for your friend's Streamlit dashboard

### The thread_id design

Every `graph.invoke()` call needs a unique `thread_id` in the config. `MemorySaver` uses this to identify which run's state to load and save. We use `issue_key + timestamp` (e.g. `"DE2-1_2026-07-21T10:30:00"`) to ensure uniqueness. This means:
- Multiple runs on the same proposal never collide
- Re-running the same proposal with a new timestamp starts a fresh run

### Why reversed() for extracting results?

The last message in `state["messages"]` is always the LLM's closing comment ("This is the end of the assessment."). The tool result messages we want (`write_compliance_verdict`, `get_trigger_words`, `llm_assessment`) are buried earlier in the list. `reversed()` lets us search backwards from the end and `break` as soon as we find each one.

Tool result messages have a `.name` attribute set to the tool function name. Regular LLM messages do not have this attribute. `hasattr(message, "name")` is how we tell them apart.

### The fix is always PENDING HUMAN REVIEW

The `fix` field in the result dictionary is always the string `"PENDING HUMAN REVIEW"`. The agent does not generate remediation suggestions. The human reviewer reads the findings in Jira or on the dashboard and decides what action to take. This is a deliberate design decision grounded in EU AI Act Article 14.

### Scalability for Streamlit

Your friend can call `assess_proposal(p)` from Streamlit with `GATE_MODE = "auto"` set before the call. Everything else is identical. The JSON result dictionary has a consistent structure he can always rely on for rendering the dashboard.

In [ ]:
def _extract_results(result_state):
    # Walk backwards through message history to extract tool results
    # Tool result messages have .name set to the tool function name
    # We search backwards so we find the most recent call of each tool
    policy_id     = None
    severity      = None
    reason        = None
    trigger_words = None
    llm_narrative = None

    for message in reversed(result_state["messages"]):
        if not hasattr(message, "name") or not message.name:
            continue
        if message.name == "write_compliance_verdict" and policy_id is None:
            for line in message.content.split("\n"):
                if line.startswith("POLICY VIOLATED"):
                    policy_id = line.split(":", 1)[1].strip()
                elif line.startswith("SEVERITY"):
                    severity = line.split(":", 1)[1].strip()
                elif line.startswith("REASON"):
                    reason = line.split(":", 1)[1].strip()
        elif message.name == "get_trigger_words" and trigger_words is None:
            trigger_words = message.content
        elif message.name == "llm_assessment" and llm_narrative is None:
            llm_narrative = message.content

    return policy_id, severity, reason, trigger_words, llm_narrative


def assess_proposal(p):
    global GATE_MODE
    print(f"\nAssessing {p['issue_key']}: {p['summary']}")

    if not p["description"]:
        print("  No description found. Skipping.")
        return None

    # ── Signal 1 ──────────────────────────────────────────────────────────────
    proba = float(classifier_pipeline.predict_proba([p["description"]])[0][1])
    y_pred = int(classifier_pipeline.predict([p["description"]])[0])
    print(f"  Signal 1: {'Red Flag' if y_pred == 1 else 'Compliant'} (proba={round(proba,4)})")

    timestamp = datetime.datetime.now().isoformat(timespec="seconds")

    if y_pred == 0:
        # ── Compliant: no LLM needed ──────────────────────────────────────────
        result = {
            "issue_key"    : p["issue_key"],
            "summary"      : p["summary"],
            "y_pred"       : 0,
            "proba"        : round(proba, 4),
            "prediction"   : "Compliant",
            "policy_id"    : None,
            "severity"     : None,
            "trigger_words": None,
            "reason"       : None,
            "fix"          : None,
            "llm_narrative": None,
            "final_decision": "Compliant -- no RAI violation detected.",
            "human_approved": True,
            "timestamp"    : timestamp,
        }
        post_jira_comment(p["issue_key"], (
            f"COMPLIANCE SENTINEL ASSESSMENT\n"
            f"{'='*40}\n"
            f"Issue      : {p['issue_key']}\n"
            f"Prediction : Compliant\n"
            f"Probability: {round(proba,4)}\n"
            f"Decision   : No RAI violation detected. No action required.\n"
            f"Assessed at: {timestamp}"
        ))
        return result

    # ── Signal 2: LLM agent ───────────────────────────────────────────────────
    # thread_id = issue_key + timestamp ensures uniqueness across multiple runs
    config = {"configurable": {"thread_id": f"{p['issue_key']}_{timestamp}"}}
    initial_state = {
        "proposal_text" : p["description"],
        "y_pred"        : 0,
        "proba"         : 0.0,
        "severity"      : "",
        "messages"      : [HumanMessage(content=p["description"])],
        "human_approved": False,
        "final_decision": "",
    }

    result_state = graph.invoke(initial_state, config=config)

    # Extract tool results from message history
    policy_id, severity, reason, trigger_words, llm_narrative = _extract_results(result_state)

    result = {
        "issue_key"    : p["issue_key"],
        "summary"      : p["summary"],
        "y_pred"       : result_state["y_pred"],
        "proba"        : result_state["proba"],
        "prediction"   : "Red Flag",
        "policy_id"    : policy_id,
        "severity"     : severity,
        "trigger_words": trigger_words,
        "reason"       : reason,
        "fix"          : "PENDING HUMAN REVIEW",
        "llm_narrative": llm_narrative,
        "final_decision": result_state["final_decision"],
        "human_approved": result_state["human_approved"],
        "timestamp"    : timestamp,
    }

    # Build and post Jira comment
    comment_lines = [
        "COMPLIANCE SENTINEL ASSESSMENT",
        "=" * 40,
        f"Issue           : {p['issue_key']}",
        f"Prediction      : Red Flag",
        f"Probability     : {result_state['proba']}",
        f"Policy violated : {policy_id or 'Unknown'}",
        f"Severity        : {severity or 'Unknown'}",
        "",
        "REASON:",
        reason or "See full assessment.",
        "",
        "RECOMMENDED FIX: PENDING HUMAN REVIEW",
        "The human reviewer must decide the appropriate remediation.",
        "",
        "LLM NARRATIVE:",
        llm_narrative or "Not available.",
        "",
        "TRIGGER WORDS:",
        trigger_words or "Not available.",
        "",
        f"Decision        : {result_state['final_decision']}",
        f"Assessed at     : {timestamp}",
        "",
        "NOTE: Generated by the Autonomous Compliance Sentinel.",
        "This assessment identifies the concern. The fix is for the human reviewer to decide.",
    ]
    post_jira_comment(p["issue_key"], "\n".join(comment_lines))

    # Print results in notebook
    print("\n--- FINAL DECISION ---")
    print(result["final_decision"])
    print(f"Classifier prediction : {'Red Flag' if result['y_pred']==1 else 'Compliant'}")
    print(f"Probability           : {result['proba']}")
    print(f"Human approved        : {result['human_approved']}")
    print("\n--- COMPLIANCE FINDINGS ---")
    print(f"POLICY VIOLATED : {policy_id or 'Unknown'}")
    print(f"SEVERITY        : {severity or 'Unknown'}")
    print(f"REASON          : {reason or 'Not available'}")
    print(f"FIX             : PENDING HUMAN REVIEW")
    print("\n--- LLM NARRATIVE ---")
    print(llm_narrative or "Not available.")

    return result

print("assess_proposal() and _extract_results() defined.")

### Step 12a -- Save Results to JSON

`save_result(result)` appends the result to `compliance_results.json`. If an entry for the same `issue_key` already exists (from a previous run), it is replaced with the new result rather than duplicated.

**Why JSON?**
Your friend's Streamlit dashboard reads this file at startup to populate the results table. JSON is the simplest format for this: no database needed, no schema migrations, no connection strings. `json.dump(..., indent=2, ensure_ascii=False)` produces a human-readable file that can also be inspected manually in any text editor.

**The deduplication logic:**
```python
existing = [r for r in existing if r["issue_key"] != result["issue_key"]]
existing.append(result)
```
This list comprehension filters out any existing entry for this `issue_key`, then appends the new result. The file always contains at most one entry per Jira issue, always the most recent run.

In [ ]:
OUTPUT_PATH = r"C:\MyGithubSpace\Data-Ethics\Works\Week 4\compliance_results.json"

def save_result(result):
    if result is None:
        return
    # Load existing results if file exists, start fresh if not
    try:
        with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
            existing = json.load(f)
    except FileNotFoundError:
        existing = []
    # Remove old entry for this issue key if it exists, then append new result
    existing = [r for r in existing if r["issue_key"] != result["issue_key"]]
    existing.append(result)
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(existing, f, indent=2, ensure_ascii=False)
    print(f"  Saved to {OUTPUT_PATH}")

print("save_result() defined.")

---
## Step 13 -- Select and Assess a Proposal

### How to use this cell

1. **Set `GATE_MODE`** at the top before running:
   - `"interactive"` -- pauses for your input on High severity findings (default, for notebook use)
   - `"auto"` -- auto-approves without pausing (for Streamlit or when running multiple proposals)

2. **Run the cell** -- it prints the numbered list of all proposals fetched from Jira in Step 7

3. **Type a number** when prompted -- the agent runs on that proposal only

4. **Type APPROVE or REJECT** if the proposal is High severity and you are in interactive mode

5. **Check Jira** -- the findings are posted as a comment on the issue you selected

6. **Check the JSON file** -- the result is saved to `compliance_results.json` for the Streamlit dashboard

### Why we pick one proposal at a time

The Groq free tier has a token per minute (TPM) rate limit of 6000 tokens. Each proposal assessment uses roughly 2000-2500 tokens across 5-6 tool calls. Running all 10 proposals back to back would hit the rate limit after the 2nd or 3rd proposal and crash with a `429 RateLimitError`.

Picking one at a time lets you pace the assessments without hitting the limit. If you need to run all proposals in batch mode, add `time.sleep(30)` between calls.

### thread_id behaviour

The `thread_id` is set to `issue_key + timestamp` inside `assess_proposal()`. This means every run of this cell on the same proposal creates a fresh graph run. LangGraph will not try to resume a previous run.

### What `proposals` contains

`proposals` was fetched from Jira in Step 7 and has been available since then. It is a list of dictionaries, one per Jira issue, each with `issue_key`, `summary`, and `description`. The numbered list printed here comes directly from that list.

In [ ]:
# Set gate mode before running
# "interactive" -- pauses for human input on High severity (for notebook use)
# "auto"        -- auto-approves, for Streamlit or scripted use
GATE_MODE = "interactive"

# Print the numbered proposal list from Jira
print("\nProposals fetched from Jira:")
print("-" * 60)
for i, p in enumerate(proposals):
    print(f"{i+1:>2}. {p['issue_key']}: {p['summary']}")
print("-" * 60)

# Ask user to pick one
choice = input("\nEnter the number of the proposal to assess: ").strip()

try:
    index = int(choice) - 1
    if index < 0 or index >= len(proposals):
        print(f"Invalid number. Please enter a number between 1 and {len(proposals)}.")
    else:
        selected = proposals[index]
        result = assess_proposal(selected)
        save_result(result)
except ValueError:
    print("Please enter a number, not text.")